# E-Mail Spam Detection System Using Machine Learning

This Jupyter notebook implements the complete 7-step Machine Learning pipeline:
1. Dataset loading & validation (`data/spam_dataset.csv`)
2. NLP text preprocessing (HTML removal, URL/currency normalization, stop-word removal)
3. TF-IDF Vectorization (unigrams + bigrams, fitted strictly on training data)
4. Training Multinomial Naive Bayes, Logistic Regression, and Platt-Calibrated Linear SVM
5. Evaluation (Accuracy, Precision, Recall, F1-Score, Confusion Matrix, ROC-AUC, 5-Fold CV)
6. Model selection & persistence with `joblib`
7. Real-time inference verification

In [ ]:
import re
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

df = pd.read_csv('data/spam_dataset.csv')
print('Dataset shape:', df.shape)
print(df['label'].value_counts())

In [ ]:
STOP_WORDS = {'i','me','my','we','our','you','your','he','she','it','they','the','a','an','and','or','but','if','of','at','by','for','with','to','from','in','out','on','is','are','was','were','be','been','have','has','had','do','does','did','subject','hi','hello','dear','thanks'}

def preprocess_text(raw_text: str) -> str:
    text = re.sub(r'<[^>]+>', ' ', str(raw_text))
    text = re.sub(r'https?://\S+|www\.\S+', ' urltoken ', text, flags=re.IGNORECASE)
    text = re.sub(r'\$\s*\d[\d,.]*', ' moneytoken ', text)
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return ' '.join(t for t in text.split(' ') if len(t) >= 2 and t not in STOP_WORDS)

df['clean_text'] = df['text'].apply(preprocess_text)
df['y'] = df['label'].str.lower().map({'spam': 1, 'ham': 0})

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['y'], test_size=0.2, random_state=42, stratify=df['y']
)

In [ ]:
models = {
    'Multinomial Naive Bayes': MultinomialNB(alpha=0.5),
    'Logistic Regression': LogisticRegression(C=1.5, max_iter=1000, random_state=42),
    'Linear SVM (Calibrated)': CalibratedClassifierCV(LinearSVC(C=1.0, random_state=42), cv=3, method='sigmoid')
}

results = []
pipelines = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in models.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=3500, sublinear_tf=True)),
        ('clf', clf)
    ])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    cv_acc = np.mean(cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy'))
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probs),
        'CV-5 Acc': cv_acc
    })
    print(f'=== {name} ===')
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds, target_names=['HAM', 'SPAM']))

pd.DataFrame(results)

In [ ]:
best_row = max(results, key=lambda r: (r['F1'], r['ROC-AUC']))
best_name = best_row['Model']
print('Selected best model:', best_name)

Path('backend/ml').mkdir(parents=True, exist_ok=True)
joblib.dump(pipelines[best_name], 'backend/ml/spam_pipeline.joblib')

sample_emails = [
    'URGENT: Your bank account is suspended. Verify SSN and password immediately!',
    'Hi team, let us move Thursday architecture review to 3:30 PM.'
]
for email in sample_emails:
    cleaned = preprocess_text(email)
    prob = pipelines[best_name].predict_proba([cleaned])[0][1]
    print(email[:55], '->', 'SPAM' if prob >= 0.5 else 'HAM', f'({prob:.4f})')